# IOAI — 2025 Lab Lab4 Multimodality Embeddings (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!pip -q install hdbscan bokeh
print('데이터(lab4data.zip)는 노트북 셀이 공개 GCS 에서 자동 다운로드합니다. CLIP 은 transformers 로 자동 로드.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Lab 4 - Multimodality and embeddings

In [ ]:
# 추가 의존성(군집화 HDBSCAN + 인터랙티브 시각화 bokeh)
!pip install -q hdbscan bokeh


## Load data

In [ ]:
!curl https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/lab4data.zip -o lab4data.zip

In [ ]:
!unzip -n lab4data.zip

## Load model

Load and initialize a model using the huggingface transformers or huggingface hub library. Use this specific model: https://huggingface.co/openai/clip-vit-base-patch32.

In [ ]:
# Your work below
from transformers import CLIPProcessor, CLIPModel

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval();

## Zero-shot classification with CLIP

Zero-shot classification means to perform classification while having seen zero images per class sample before hand. 

Use CLIP to classify the images provided into bears and flowers without doing any finetuning! Show classification results through a confusion matrix.

In [ ]:
# Your work below
import torch
from PIL import Image
from pathlib import Path
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
classes = []
images = []
image_paths = []
labels = []

for dir_ in Path("./lab4data").iterdir():
    if not dir_.is_dir():  # dir_ is a Path object
        continue

    class_ = dir_.name  # dir_.name is the final path component
    classes.append(class_)
    for image_dir in dir_.glob("*.jpg"):
        images.append(Image.open(image_dir).convert("RGB"))
        image_paths.append(str(image_dir))
        labels.append(len(classes) - 1)

In [ ]:
classes  # based on folder names

In [ ]:
len(images), len(labels)

In [ ]:
inputs = processor(text=classes, images=images, return_tensors="pt", padding=True)

with torch.no_grad():
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image  # shape: [len(images), len(classes)]
    probs = logits_per_image.softmax(dim=1)

predictions = probs.argmax(dim=1)
predictions

In [ ]:
torch.tensor(labels)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(labels, predictions), display_labels=classes)
disp.plot(cmap="Blues")

All correct!

## Feature extraction using ResNet

Zero-shot classification works as it does, because it makes use of a well-conditioned latent space where all images are semantically separable. Prove that you can do something similar to the previous section using any image backbone!

Use the same ResNet backbone in the past few labs (Resnet34 w/o last two layers) and perform feature extraction for each image. Show me how many subclasses of bears are there in this dataset. 

Hint: Week5 lecture.

In [ ]:
# Your work below
import matplotlib.pyplot as plt
from torchvision import models, transforms
from torchvision.models import ResNet34_Weights
from sklearn.preprocessing import StandardScaler
from hdbscan import HDBSCAN
from sklearn.decomposition import PCA

import base64
from io import BytesIO
from bokeh.palettes import Category10
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.io import output_notebook
output_notebook()

In [ ]:
resnet34 = models.resnet34(weights=ResNet34_Weights.DEFAULT)
resnet34 = torch.nn.Sequential(*list(resnet34.children())[:-2])  # Remove last two layers

In [ ]:
resnet_processor = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
batch_bears = []
bear_pil_images = []

for i in range(len(images)):
    if labels[i] != 0:
        continue

    batch_bears.append(resnet_processor(images[i]))
    bear_pil_images.append(images[i])

batch_bears = torch.stack(batch_bears)

In [ ]:
batch_bears.shape

In [ ]:
resnet34.eval()
with torch.no_grad():
    features = resnet34(batch_bears)
features = features.detach().cpu().numpy()
features.shape

In [ ]:
features = features.reshape(features.shape[0], -1)
features.shape

In [ ]:
scaler = StandardScaler()
pca = PCA(n_components=2, random_state=42)

In [ ]:
features_scaled = scaler.fit_transform(features)
features_scaled_2d = pca.fit_transform(features_scaled)
features_scaled_2d.shape

In [ ]:
hdbscan = HDBSCAN(min_cluster_size=4)
predicted_subclasses = hdbscan.fit_predict(features_scaled_2d)
predicted_subclasses

In [ ]:
def encode_image(img):  # PIL Image to data image URI
    buffer = BytesIO()
    img.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"

In [ ]:
unique_classes = list(set(predicted_subclasses))
palette = Category10[10]  # Use a larger palette if more than 10 classes
class_to_color = {cls: palette[i % 10] for i, cls in enumerate(unique_classes)}
colors = [class_to_color[c] for c in predicted_subclasses]

source = ColumnDataSource(data={
    "x": features_scaled_2d[:, 0],
    "y": features_scaled_2d[:, 1],
    "classes": colors,
    "imgs": list(map(encode_image, bear_pil_images))
})

fig = figure(title="Bear Feature Clusters in PCA Space")
hover = HoverTool(tooltips="""
    <div>
        <img src=@imgs height="100" style="border: 1px solid black;"/>
    </div>
""")
fig.add_tools(hover)

for cls in unique_classes:
    idx = [i for i, c in enumerate(predicted_subclasses) if c == cls]
    source = ColumnDataSource(data={
        "x": features_scaled_2d[idx, 0],
        "y": features_scaled_2d[idx, 1],
        "imgs": [encode_image(bear_pil_images[i]) for i in idx]
    })
    fig.scatter("x", "y", size=10, color=class_to_color[cls], legend_label=f"Class {cls}" if cls > -1 else "Outliers", source=source)

show(fig)

Hover over the dots to view their corresponding images.

There are two distinct clusters: the one on the left represents teddy bears, while the cluster on the right represents real bears, including pandas, polar bears, and brown bears.

## Debiasing vectors in latent space

Realize that the functionality of CLIP rests upon its capability to map both images and text into the same latent space. Learning how to navigate latent spaces is key to making best use of them. Fortunately this is quite learnable. Each instance of data, in this case images, are represented by vectors of size (N, ) where N is the dimension of the latent space. If you can learn Euclid's art of geometry, you can navigate latent spaces.

This 2016 paper, "Man is to Computer Programmer as Woman is to Homemaker? Debiasing Word Embeddings" (https://arxiv.org/pdf/1607.06520) shows that:

1. Vector differences between word embeddings can represent relationships between words,
2. Gender biases derived from linguistic use of specific terms can be quantified in these words!

If you take the word embedding vector of man and minus the word embedding vector of woman, this difference is numerically close to the word embedding vectors of king minus queen. This is a more innocuous example as king and queen are historically gendered roles and are appropriately accurate. The paper discovered that this gender bias is baked into a lot more words that might not accurately reflect modern progressive views on gender.

Before we go further, let me explain some small differences in terminology. 

- This article used word embeddings derived from the word2vec embedder. Word2vec is trained by calculating the co-occurence of words in a corpus. If two words are seen together very often, they will have embeddings that are close, and vice versa. CLIP's embeddings are trained differently, and can allow embedding more than one word.
- Note that the terminology for many things in machine learning are still fast and loose, so use of terms are not very standardized among practitioners. In this case, embedding space and latent space are very very similar with small differences! On one hand, we describe the vector space where data points (in this case images) are transformed into vector representations where the distance among items reflect their semantic differences. On the other hand, any output of an intermediate layer in any neural network can be said to be projecting into a latent space. In real life usage, I think these terms are mixed up enough that it suffices for you to know that they tend to mean the same thing.

Let us attempt to replicate the findings of this paper using CLIP's text embeddings. Calculate a measure of gender inclination for the following phrases, then perform debiasing afterwards and show that the debiased vector shows perfectly neutral gender inclination.

- "A modest pope"
- "A haughty unicorn"
- "A fanciful flower"
- "A ferocious bear"

You can use any method to measure gender inclination, as long as it reflects how much closer is the phrase to the concept of male vs to the concept of female.

In [ ]:
# Your work below
import seaborn as sns
import torch.nn.functional as F

In [ ]:
def to_embeddings(text):
    inputs = processor(text=text, return_tensors="pt", padding=True, add_special_tokens=True)
    with torch.no_grad():
        embeddings = model.get_text_features(**inputs)
    return embeddings.to(torch.float64)

In [ ]:
genders = ["a photo of a woman", "a photo of a man"]  # A more descriptive one works better
gender_embeddings = to_embeddings(genders)
gender_embeddings.shape

In [ ]:
texts = ["A modest pope", "A haughty unicorn", "A fanciful flower", "A ferocious bear"]
biased_embeddings = to_embeddings(texts)
biased_embeddings.shape

In [ ]:
def show_inclination(target_embeddings):
    gender_similarity = torch.vstack([F.cosine_similarity(target_embeddings, gender_embeddings[i]) for i in range(len(genders))]).T

    for i, row in enumerate(gender_similarity):
        print(f"'{texts[i]}' is inclined towards '{genders[row.argmax()]}' by {row.max() - row.min():.8f}.")

    sns.heatmap(gender_similarity, annot=True, fmt=".8f", xticklabels=genders, yticklabels=texts, cmap="Blues", vmin=0.5, vmax=1)
    plt.title("Inclination of CLIP's Embeddings")
    plt.xlabel("class")
    plt.ylabel("phrase")

In [ ]:
show_inclination(biased_embeddings)

In the extreme case, the phrase 'a queen' shows a cosine similarity difference of 0.03 between classes. This value can serve as a reference of how inclined the above phrases are. 'A modest pope' and 'a ferocious bear' are quite biased, as a clear barrier in hues can be seen across classes. The other two are okay.

### Identify bias subspace

We start by specifying $m$ defining pairs,

$$
D_i = \{(w_i^+, w_i^-)\}_{i=1}^m
$$ 

where each pair consists of a “female” word $w_i^+$ and its “male” counterpart $w_i^-$.

We then map each word $w_i$ to its embedding vector $\mathbf{e}(w_i)\in\mathbb{R}^d$. Each of the embeddings is adjusted by

$$
\mathbf{e}(w_i) \leftarrow \mathbf{e}(w_i) - \frac{1}{2} \sum w_i.
$$

Stacking these yields a matrix,

$$
\mathbf{W} = 
\begin{bmatrix}
\mathbf{e}(w_1^+)^\top \\
\mathbf{e}(w_1^-)^\top \\
\vdots \\
\mathbf{e}(w_m^+)^\top \\
\mathbf{e}(w_m^-)^\top \\
\end{bmatrix}
\quad\in\mathbb{R}^{2m\times d}.
$$

In [ ]:
D = [["she", "he"], ["daughter", "son"], ["her", "his"], 
     ["mother", "father"], ["Mary", "John"], ["niece", "nephew"],
     ["girl", "boy"], ["herself", "himself"], ["aunt", "uncle"],
     ["bride", "groom"], ["queen", "king"], ["madam", "sir"]]
W = torch.vstack([to_embeddings(Di) for Di in D]).reshape(len(D), len(D[0]), -1)
W.shape

In [ ]:
W_centered = (W - W.mean(dim=1, keepdim=True)).reshape(-1, W.shape[-1])
W_centered.shape

The covariance matrix is given by

$$
\mathbf{C} = \frac{1}{2m - 1} \mathbf{W}^\top \mathbf{W}
\quad\in\mathbb{R}^{d\times d}.
$$

Since $\mathbf{C}$ is symmetric, it can be decomposed as

$$
\mathbf{C} = \mathbf{Q} \mathbf{\Lambda} \mathbf{Q}^\top,
$$

where:

- $\mathbf{Q}$ is an orthogonal matrix containing the eigenvectors of $\mathbf{C}$.
- $\mathbf{\Lambda}$ is a diagonal matrix with the eigenvalues of $\mathbf{C}$ on the diagonal, sorted in descending order.

The first eigenvector in $\mathbf{Q}$ is the vector that spans our bias subspace.

Overall, these steps essentially correspond to PCA. We will use `sklearn.decomposition.PCA` to simplify the process.

In [ ]:
pca1 = PCA()
pca1.fit(W_centered)
b = torch.tensor(pca1.components_[0])  # Take one component only, this is the bias subspace
b.shape

If you prefer a manual one, this code snippet is equivalent to the above:

```python
C = W_centered.T @ W_centered / (W_centered.shape[0] - 1)

eigenvalues, eigenvectors = torch.linalg.eigh(C)

sorted_indices = torch.argsort(eigenvalues, descending=True)
eigenvalues = eigenvalues[sorted_indices]
eigenvectors = eigenvectors[:, sorted_indices]

b = eigenvectors[:, 0]
```

It has been tested to produce the same output `b`.

### Neutralize

This part is done by enforcing all neutral words to lie on the hyperplane orthogonal to the bias direction.

Let:

- $\mathbf{V} \in \mathbb{R}^{n \times d}$ be a matrix of **biased embeddings** (each row is an embedding),
- $\mathbf{b} \in \mathbb{R}^d$ be the **bias direction**, a unit vector.

We project each row vector $\mathbf{v}_i$ in $\mathbf{V}$ onto $\mathbf{b}$,

$$
\text{proj}_{\mathbf{b}}(\mathbf{V}) = \mathbf{V}\mathbf{b}\mathbf{b}^\top
\quad\in\mathbb{R}^{n \times d}.
$$



In [ ]:
emb_proj = (biased_embeddings @ b)[:, None] * b[None, :]
emb_proj.shape

Then, we subtract the projection from the original embeddings and normalize.

In [ ]:
unbiased_embeddings = biased_embeddings - emb_proj
unbiased_embeddings /= unbiased_embeddings.norm(dim=1, keepdim=True)
unbiased_embeddings.shape

In [ ]:
show_inclination(unbiased_embeddings)

The differences are now one-tenth of what they were before.

## EX: Most opposite word

Find a single word from CLIP's vocabulary that will give the shortest distance with the embedding of the phrase: "shortest path between two points". Make sure that the word is represented by its own token when passed through CLIP's tokenizer. In other words, when passed through CLIP's tokenizer, we should only see three tokens: token 49406, the word's own token, and token 49407.

In [ ]:
# Your work below
import nltk
nltk.download('words')
from nltk.corpus import words
from transformers import CLIPTokenizer 

english_words = set(words.words())
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
phrase = "shortest path between two points"
phrase_embedding = to_embeddings(phrase).squeeze(0)
phrase_embedding.shape

In [ ]:
# Criteria: English word with only 3 tokens
vocab = tuple(filter(lambda word: word in english_words and tokenizer(word, return_tensors="pt")["input_ids"].shape[1] == 3, english_words))
len(english_words), len(vocab)

In [ ]:
vocab_embeddings = to_embeddings(vocab)
vocab_embeddings.shape

In [ ]:
distances = F.cosine_similarity(vocab_embeddings, phrase_embedding.unsqueeze(0), dim=1)
distances.shape

In [ ]:
max_idx = distances.argmax().item()
vocab[max_idx], tokenizer(vocab[max_idx])["input_ids"]

The answer is 'route' with token ID 5261.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv', 'submission.zip', 'submission.jsonl', 'submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)